# 07 Pipe 3 - agent scientometrics rebuild

Notebook canonico da Camada 3 para:

1. carregar os artefatos de `06_pipe2_retriever_analytics`;
2. adicionar o baseline `BM25`;
3. montar bundles de evidencia por query e por retriever;
4. produzir saídas experimentais replicáveis para `core` e `holdout`;
5. opcionalmente gerar resumos grounded por LLM sem quebrar a rastreabilidade.

## Observacao metodologica

- O banco de queries continua derivado do `core`.
- O `holdout` espelha a mesma familia de aplicacoes.
- Sem chave de API, o notebook ainda produz o experimento retrieval-only completo.


In [ ]:
# ============================================================
# Instalacao de dependencias para Colab
# ============================================================
!pip install -U -q pip setuptools wheel
!pip install -U -q numpy==2.1.2 scipy==1.14.1 pandas==2.2.2 scikit-learn==1.5.2
!pip install -U -q faiss-cpu sentence-transformers==2.7.0 transformers==4.45.2 rank-bm25==0.2.2
!pip install -U -q openai==1.* pydantic openpyxl

print("Dependencias instaladas.")


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import json
import math
import os
import re
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from openai import OpenAI
from rank_bm25 import BM25Okapi

pd.set_option("display.max_columns", 220)
pd.set_option("display.max_colwidth", 260)

print("OPENAI_API_KEY presente:", bool(os.getenv("OPENAI_API_KEY")))


In [ ]:
# ============================================================
# Configuracao geral
# ============================================================

DRIVE_ROOT = Path("/content/drive/MyDrive/Unicamp")
PROJECT_ROOT = DRIVE_ROOT / "artigo bibliometria" / "grounded-scientometrics-solarphysics-retrieval"
DATA_ROOT = DRIVE_ROOT / "artigo bibliometria" / "base de dados" / "Artigo_Bibliometria Base Bruta" / "BASES_UNIFICADAS_POR_TEMA"

TARGET_CORPUS = "ML_Multimodal"
READ_STAGE = "06_pipe2_retriever_analytics"
WRITE_STAGE = "07_pipe3_agent_scientometrics"
PERIODS = ["core", "holdout"]
TOPK = 20
MAX_LLM_QUERIES = 30
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
USE_OPENAI_IF_AVAILABLE = True

PIPE2_ROOT = DATA_ROOT / TARGET_CORPUS / "04_rebuild_outputs" / READ_STAGE
WRITE_ROOT = DATA_ROOT / TARGET_CORPUS / "04_rebuild_outputs" / WRITE_STAGE

assert PIPE2_ROOT.exists(), f"PIPE2_ROOT nao encontrado: {PIPE2_ROOT}"
print("PIPE2_ROOT =", PIPE2_ROOT)
print("WRITE_ROOT =", WRITE_ROOT)


In [ ]:
# ============================================================
# Saidas, logs e helpers
# ============================================================

RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")
PIPE_START_TS = time.time()


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


ensure_dir(WRITE_ROOT)
ensure_dir(WRITE_ROOT / "tables")
ensure_dir(WRITE_ROOT / "reports")
for period in PERIODS:
    ensure_dir(WRITE_ROOT / period / "tables")
    ensure_dir(WRITE_ROOT / period / "reports")

GLOBAL_LOG_DIR = ensure_dir(PROJECT_ROOT / "outputs" / "camada3_logs")
GLOBAL_LOG_FILE = GLOBAL_LOG_DIR / f"07_pipe3_{RUN_TS}.txt"


def elapsed_seconds() -> float:
    return time.time() - PIPE_START_TS


def fmt_seconds(seconds: float) -> str:
    seconds = int(round(seconds))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    if h:
        return f"{h:02d}:{m:02d}:{s:02d}"
    return f"{m:02d}:{s:02d}"


def log(message: str) -> None:
    now = datetime.now().strftime("%H:%M:%S")
    prefix = f"[{now} | +{fmt_seconds(elapsed_seconds())}]"
    line = f"{prefix} {message}"
    print(line, flush=True)
    with open(GLOBAL_LOG_FILE, "a", encoding="utf-8") as fh:
        fh.write(line + "\n")


def stage_banner(title: str) -> None:
    bar = "=" * 96
    log(bar)
    log(title)
    log(bar)


In [ ]:
# ============================================================
# Leitura dos artefatos do Pipe 2
# ============================================================

stage_banner("LEITURA DOS ARTEFATOS DO PIPE 2")

query_bank = pd.read_csv(PIPE2_ROOT / "tables" / "core_gap_query_bank.csv")
semantic_hits = pd.read_csv(PIPE2_ROOT / "tables" / "semantic_query_hits.csv")
semantic_metrics = pd.read_csv(PIPE2_ROOT / "tables" / "semantic_query_metrics.csv")
white_space = pd.read_csv(PIPE2_ROOT / "tables" / "white_space_candidates.csv")

period_docs = {
    period: pd.read_csv(PIPE2_ROOT / period / "tables" / f"{TARGET_CORPUS}_{period}_docs_master.csv")
    for period in PERIODS
}

log(f"[load] queries={len(query_bank)} | semantic_hits={len(semantic_hits)} | white_space={len(white_space)}")
display(query_bank.head(10))
display(white_space.head(10))


In [ ]:
# ============================================================
# Baseline BM25
# ============================================================

stage_banner("BM25")


def tokenize_for_bm25(text: str) -> list[str]:
    text = re.sub(r"[^a-z0-9\s]", " ", str(text).lower())
    return [tok for tok in text.split() if tok]


bm25_models = {}
bm25_tokens = {}
for period in PERIODS:
    docs = period_docs[period].copy()
    docs["bm25_text"] = (docs["title"].fillna("") + " " + docs["abstract"].fillna("") + " " + docs["keywords"].fillna("")).astype(str)
    tokens = [tokenize_for_bm25(text) for text in docs["bm25_text"].tolist()]
    bm25_models[period] = BM25Okapi(tokens)
    bm25_tokens[period] = tokens
    log(f"[bm25] {period} | docs={len(docs)}")


bm25_rows = []
for period in PERIODS:
    docs = period_docs[period]
    total_queries = len(query_bank)
    for q_idx, q_row in enumerate(query_bank.itertuples(index=False), start=1):
        scores = bm25_models[period].get_scores(tokenize_for_bm25(q_row.query_text))
        top_idx = np.argsort(scores)[::-1][:TOPK]
        hits = docs.iloc[top_idx].copy().reset_index(drop=True)
        hits["rank"] = np.arange(1, len(hits) + 1)
        hits["score"] = [float(scores[i]) for i in top_idx]
        hits["period"] = period
        hits["retriever"] = "bm25"
        hits["query_id"] = q_row.query_id
        hits["query_text"] = q_row.query_text
        hits["source_corpus"] = q_row.source_corpus
        bm25_rows.append(hits)
        if q_idx == 1 or q_idx % 10 == 0 or q_idx == total_queries:
            log(f"[bm25] {period} | query {q_idx}/{total_queries}")

bm25_hits = pd.concat(bm25_rows, ignore_index=True)
bm25_hits.to_csv(WRITE_ROOT / "tables" / "bm25_query_hits.csv", index=False)
display(bm25_hits.head(20))


In [ ]:
# ============================================================
# Bundles de evidencia e metricas por retriever
# ============================================================

stage_banner("BUNDLES DE EVIDENCIA")

semantic_hits = semantic_hits.rename(columns={"retriever": "retriever"})
all_hits = pd.concat([semantic_hits, bm25_hits], ignore_index=True)
all_hits.to_csv(WRITE_ROOT / "tables" / "retrieval_hits_all.csv", index=False)

metric_rows = []
bundle_rows = []

for period in PERIODS:
    ws_period = white_space[white_space["period"] == period].copy()
    ws_lookup = ws_period.set_index("query_id") if len(ws_period) else None
    for retriever in ["generic", "specialized", "bm25"]:
        sub = all_hits[(all_hits["period"] == period) & (all_hits["retriever"] == retriever)].copy()
        for query_id, q_sub in sub.groupby("query_id", sort=False):
            q_sub = q_sub.sort_values("rank").copy()
            metric_rows.append(
                {
                    "period": period,
                    "retriever": retriever,
                    "query_id": query_id,
                    "query_text": q_sub["query_text"].iloc[0],
                    "source_corpus": q_sub["source_corpus"].iloc[0],
                    "top1_score": float(q_sub["score"].iloc[0]),
                    "mean_topk_score": round(float(q_sub["score"].head(TOPK).mean()), 4),
                    "mean_topk_citations": round(float(pd.to_numeric(q_sub["citations"], errors="coerce").fillna(0).head(TOPK).mean()), 2),
                    "recent_share_topk": round(float(pd.to_numeric(q_sub["year"], errors="coerce").fillna(0).head(TOPK).ge(2023).mean()), 4),
                    "distinct_sources_topk": int(q_sub["source"].fillna("").head(TOPK).nunique()),
                    "white_space_candidate_score": float(ws_lookup.loc[query_id, "candidate_score"]) if ws_lookup is not None and query_id in ws_lookup.index else np.nan,
                }
            )

            bundle_rows.append(
                {
                    "period": period,
                    "retriever": retriever,
                    "query_id": query_id,
                    "query_text": q_sub["query_text"].iloc[0],
                    "source_corpus": q_sub["source_corpus"].iloc[0],
                    "bundle_json": json.dumps(
                        q_sub.head(10)[["doc_id", "title", "year", "source", "score", "doi"]].to_dict(orient="records"),
                        ensure_ascii=False,
                    ),
                }
            )

metrics_df = pd.DataFrame(metric_rows)
bundles_df = pd.DataFrame(bundle_rows)
metrics_df.to_csv(WRITE_ROOT / "tables" / "retriever_query_metrics.csv", index=False)
bundles_df.to_csv(WRITE_ROOT / "tables" / "evidence_bundles.csv", index=False)

display(metrics_df.head(20))
display(bundles_df.head(10))


In [ ]:
# ============================================================
# Saida agent-like grounded (OpenAI opcional)
# ============================================================

stage_banner("SAIDA AGENT-LIKE")

candidate_queries = (
    white_space.sort_values(["period", "candidate_score"], ascending=[True, False])
    .groupby("period", group_keys=False)
    .head(MAX_LLM_QUERIES)
    .copy()
)

client = OpenAI() if (USE_OPENAI_IF_AVAILABLE and os.getenv("OPENAI_API_KEY")) else None
agent_rows = []

for idx, row in enumerate(candidate_queries.itertuples(index=False), start=1):
    specialized_bundle = bundles_df[
        (bundles_df["period"] == row.period)
        & (bundles_df["retriever"] == "specialized")
        & (bundles_df["query_id"] == row.query_id)
    ]
    evidence_json = specialized_bundle["bundle_json"].iloc[0] if len(specialized_bundle) else "[]"

    if client is None:
        summary = f"Retrieval-only bundle for {row.query_id}. Query: {row.query_text}"
        status = "heuristic_only"
    else:
        prompt = f"""
Voce esta auditando gaps cientometricos.
Use somente as evidencias listadas abaixo.
Retorne um paragrafo curto com:
1. gap-title curto
2. justificativa grounded
3. doc_ids usados

QUERY: {row.query_text}
EVIDENCIAS: {evidence_json}
"""
        try:
            response = client.chat.completions.create(
                model=OPENAI_MODEL,
                temperature=0,
                messages=[
                    {"role": "system", "content": "Seja objetivo e grounded nas evidencias."},
                    {"role": "user", "content": prompt},
                ],
            )
            summary = response.choices[0].message.content
            status = "openai_ok"
        except Exception as exc:
            summary = f"Falha na chamada OpenAI: {exc}"
            status = "openai_error"

    agent_rows.append(
        {
            "period": row.period,
            "query_id": row.query_id,
            "query_text": row.query_text,
            "source_corpus": row.source_corpus,
            "candidate_score": row.candidate_score,
            "agent_status": status,
            "agent_output": summary,
        }
    )

    if idx == 1 or idx % 5 == 0 or idx == len(candidate_queries):
        log(f"[agent] processed {idx}/{len(candidate_queries)}")

agent_df = pd.DataFrame(agent_rows)
agent_df.to_csv(WRITE_ROOT / "tables" / "agent_outputs.csv", index=False)
with open(WRITE_ROOT / "reports" / "agent_outputs.jsonl", "w", encoding="utf-8") as fh:
    for row in agent_rows:
        fh.write(json.dumps(row, ensure_ascii=False) + "\n")

display(agent_df.head(20))


In [ ]:
# ============================================================
# Tabelas do artigo e manifesto final
# ============================================================

stage_banner("EXPORT FINAL DO PIPE 3")

paper_summary = (
    metrics_df.groupby(["period", "retriever"], as_index=False)
    .agg(
        n_queries=("query_id", "nunique"),
        mean_top1_score=("top1_score", "mean"),
        mean_mean_topk_score=("mean_topk_score", "mean"),
        mean_topk_citations=("mean_topk_citations", "mean"),
        mean_recent_share=("recent_share_topk", "mean"),
        mean_distinct_sources=("distinct_sources_topk", "mean"),
    )
)
paper_summary.to_csv(WRITE_ROOT / "tables" / "paper_ready_retriever_summary.csv", index=False)

with pd.ExcelWriter(WRITE_ROOT / "reports" / "pipe3_paper_tables.xlsx", engine="openpyxl") as writer:
    paper_summary.to_excel(writer, sheet_name="retriever_summary", index=False)
    metrics_df.to_excel(writer, sheet_name="query_metrics", index=False)
    agent_df.to_excel(writer, sheet_name="agent_outputs", index=False)

manifest_rows = []
for path in sorted(WRITE_ROOT.rglob("*")):
    if path.is_file():
        manifest_rows.append({"artifact": str(path), "size_bytes": path.stat().st_size})

manifest_df = pd.DataFrame(manifest_rows)
manifest_df.to_csv(WRITE_ROOT / "reports" / "artifact_manifest.csv", index=False)

display(paper_summary)
display(manifest_df.head(30))
print("Arquivos finais salvos em:", WRITE_ROOT)
